# Personal AI Assistant: Colab Demo

This notebook loads a fine-tuned **LoRA adapter** for `Qwen/Qwen2.5-0.5B-Instruct` and lets you test the assistant in Google Colab.

## Before you run
1. Open this notebook in **Google Colab**.
2. Select **Runtime → Change runtime type → T4 GPU** (or another available GPU).
3. Run the cells from top to bottom.
4. When prompted, upload the `qwen-chatbot-final.zip` adapter file.

**Important:** The ZIP contains the fine-tuned adapter, not the full base model. The notebook downloads the base Qwen model from Hugging Face, so an internet connection is required. The notebook is an inference demo, not a training notebook.

In [ ]:
# Install the libraries needed to load the base model and LoRA adapter.
%pip install -q -U transformers peft accelerate safetensors

## 1. Upload your trained adapter

Upload the file named `qwen-chatbot-final.zip`. Keep the filename unchanged, or update `ADAPTER_ZIP_NAME` in the next cell.

In [ ]:
from google.colab import files

uploaded = files.upload()

ADAPTER_ZIP_NAME = "qwen-chatbot-final.zip"

if ADAPTER_ZIP_NAME not in uploaded:
    raise FileNotFoundError(
        f"Please upload {ADAPTER_ZIP_NAME}. "
        f"Files received: {list(uploaded.keys())}"
    )

print("Adapter ZIP uploaded successfully.")

## 2. Extract and check the adapter files

The adapter folder should contain `adapter_model.safetensors` and `adapter_config.json`.

In [ ]:
import os
import zipfile

ADAPTER_DIR = "/content/qwen-chatbot-final"
os.makedirs(ADAPTER_DIR, exist_ok=True)

with zipfile.ZipFile(ADAPTER_ZIP_NAME, "r") as archive:
    archive.extractall(ADAPTER_DIR)

required_files = ["adapter_model.safetensors", "adapter_config.json"]
missing_files = [
    name for name in required_files
    if not os.path.exists(os.path.join(ADAPTER_DIR, name))
]

if missing_files:
    raise FileNotFoundError(
        "The ZIP does not contain the required adapter files: "
        + ", ".join(missing_files)
    )

print("Adapter files verified:")
for name in sorted(os.listdir(ADAPTER_DIR)):
    print(" -", name)

## 3. Load the base model and attach the fine-tuned adapter

The base model is downloaded from Hugging Face. The LoRA adapter then adds the fine-tuned weights.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("\nFine-tuned model loaded successfully.")

## 4. Verify the model

In [ ]:
print("Model loaded:", model is not None)
print("Model device:", model.device)
print("Model in evaluation mode:", not model.training)

## 5. Chat with your assistant

This demo keeps the messages from the current session in `chat_history`, so replies can use earlier turns in the same session. The history is held in notebook memory only. It is not saved permanently.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful, friendly, and concise personal assistant. "
    "Respond naturally, consider the conversation context, "
    "and do not assume information the user has not provided. "
    "If important information is missing, ask a relevant follow-up question."
)

chat_history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(user_message, max_new_tokens=160):
    chat_history.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    chat_history.append({"role": "assistant", "content": reply})
    return reply

def reset_chat():
    chat_history.clear()
    chat_history.append({"role": "system", "content": SYSTEM_PROMPT})
    print("Conversation history cleared.")

## 6. Try a test message

Run this cell to check that the model generates a response. You can edit the message and run it again.

In [ ]:
print(chat("I had a tiring day with several meetings. Any suggestions to unwind?"))

## 7. Chat interactively

Run this cell and type messages in the input box. Enter `exit` to stop. Run `reset_chat()` whenever you want to start a fresh conversation.

In [ ]:
while True:
    user_text = input("You: ").strip()

    if user_text.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    if not user_text:
        continue

    print("Assistant:", chat(user_text))
    print()

## Notes and current limitations

- This notebook runs model inference. It does **not** train or fine-tune the model.
- The fine-tuned component is a LoRA adapter; the original Qwen base model is downloaded separately.
- Conversation history exists only while the current Colab runtime is active.
- This demo does not yet include a database, persistent user memory, routine prediction, a FastAPI backend, or a production user interface.
- If Colab disconnects or the runtime resets, rerun the notebook and upload the adapter ZIP again.